## Problem Statement

### Context

AllLife Bank is a US bank that has a growing customer base. The majority of these customers are liability customers (depositors) with varying sizes of deposits. The number of customers who are also borrowers (asset customers) is quite small, and the bank is interested in expanding this base rapidly to bring in more loan business and in the process, earn more through the interest on loans. In particular, the management wants to explore ways of converting its liability customers to personal loan customers (while retaining them as depositors).

A campaign that the bank ran last year for liability customers showed a healthy conversion rate of over 9% success. This has encouraged the retail marketing department to devise campaigns with better target marketing to increase the success ratio.

You as a Data scientist at AllLife bank have to build a model that will help the marketing department to identify the potential customers who have a higher probability of purchasing the loan.

### Objective

To predict whether a liability customer will buy personal loans, to understand which customer attributes are most significant in driving purchases, and identify which segment of customers to target more.

### Data Dictionary
* `ID`: Customer ID
* `Age`: Customer’s age in completed years
* `Experience`: #years of professional experience
* `Income`: Annual income of the customer (in thousand dollars)
* `ZIP Code`: Home Address ZIP code.
* `Family`: the Family size of the customer
* `CCAvg`: Average spending on credit cards per month (in thousand dollars)
* `Education`: Education Level. 1: Undergrad; 2: Graduate;3: Advanced/Professional
* `Mortgage`: Value of house mortgage if any. (in thousand dollars)
* `Personal_Loan`: Did this customer accept the personal loan offered in the last campaign? (0: No, 1: Yes)
* `Securities_Account`: Does the customer have securities account with the bank? (0: No, 1: Yes)
* `CD_Account`: Does the customer have a certificate of deposit (CD) account with the bank? (0: No, 1: Yes)
* `Online`: Do customers use internet banking facilities? (0: No, 1: Yes)
* `CreditCard`: Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes)

## Importing necessary libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy==1.26.0 pandas==1.5.3 matplotlib==3.7.1 seaborn==0.13.1 scikit-learn==1.2.2 sklearn-pandas==2.2.0 -q --user

**Note**:

1. After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab), write the relevant code for the project from the next cell, and run all cells sequentially from the next cell.

2. On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
# import libraries for data manipulation
import numpy as np
import pandas as pd

# import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# libraries for building models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KDTree
from sklearn import tree

# library for tuning different models
from sklearn.model_selection import GridSearchCV

# library to get metric scores and plot the tree
from sklearn import metrics, tree

# Library to split data
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# To get diferent metric scores
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
)

import shap

# To ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

## Loading the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('/content/drive/My Drive/Python_Course/Projects/ML_Project2/Loan_Modelling.csv')

In [ ]:
#First 5 rows
data.head()

In [ ]:
#Last 5 rows
data.tail()

## Data Overview

**Data set value details from problem statement:**

**ID:** Customer ID

**Age:** Customer’s age in completed years

**Experience:** #years of professional experience

**Income:** Annual income of the customer (in thousand dollars)

**ZIP Code:** Home Address ZIP code.

**Family:** the Family size of the customer

**CCAvg:** Average spending on credit cards per month (in thousand dollars)

**Education:** Education Level. 1: Undergrad; 2: Graduate;3: Advanced/Professional

**Mortgage:** Value of house mortgage if any. (in thousand dollars)

**Personal_Loan:** Did this customer accept the personal loan offered in the last campaign? (0: No, 1: Yes)

**Securities_Account:** Does the customer have securities account with the bank? (0: No, 1: Yes)

**CD_Account:** Does the customer have a certificate of deposit (CD) account with the bank? (0: No, 1: Yes)

**Online:** Do customers use internet banking facilities? (0: No, 1: Yes)

**CreditCard:** Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes)

* Observations
* Sanity checks

In [ ]:
#Finding shape of Data
print("Rows:", {data.shape[0]})
print("Columns:", {data.shape[1]})

In [ ]:
#Statistical summary of data
data.describe().T

In [ ]:
#Creating data copy to avoid making changes to the original
df = data.copy()

#Data Summary
df.info()

In [ ]:
#Adding a new column, "HasMortgage" to the data set.
#If the customer has taken Mortgage from AllLife bank, the column will be set to 1.
#If the customer has NOT taken Mortgage from AllLife Bank, the column will be set to 0.

for index, row in df.iterrows():
  #print(row['Mortgage'])
  if row['Mortgage'] > 0:
     df.loc[index, 'HasMortgage'] = 1
  else:
     df.loc[index, 'HasMortgage'] = 0

# Convert the new column to integer datatype
df['HasMortgage'] = df['HasMortgage'].astype('int64')

In [ ]:
#Extracting first 3 numbers of each ZIPCode to a separate column, County.

df['County'] = df['ZIPCode'].apply(lambda x: str(x)[0:3])
df['County'] = df['County'].astype(int)

#Dropping ZIP code column as County column provides required information
df.drop('ZIPCode',axis=1, inplace=True)

In [ ]:
#Adding a new column to the dataset - AllLifeBankCreditCard
#If the customer has ONLY AllLifeBankCreditCard, the value of the column will be set to 1.
#If customer has other credit cards AND does not have AllLifeBank credit card, the column will be set to 0.

for index, row in df.iterrows():
  if row['CCAvg'] > 0:
     if row['CreditCard'] == 0:
        df.loc[index, 'HasAllLifeCreditCard'] = 1
     else:
        df.loc[index, 'HasAllLifeCreditCard'] = 0
  else:
     df.loc[index, 'HasAllLifeCreditCard'] = 0
# Convert the new column to integer datatype. Handle potential NaN values by filling with 0 and then converting.
df['HasAllLifeCreditCard'] = df['HasAllLifeCreditCard'].astype('int64')

In [ ]:
df.head()

In [ ]:
df.tail()

## Exploratory Data Analysis.

- EDA is an important part of any project involving data.
- It is important to investigate and understand the data better before building a model with it.
- A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data.
- A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Questions**:

1. What is the distribution of mortgage attribute? Are there any noticeable patterns or outliers in the distribution?
2. How many customers have credit cards?
3. What are the attributes that have a strong correlation with the target attribute (personal loan)?
4. How does a customer's interest in purchasing a loan vary with their age?
5. How does a customer's interest in purchasing a loan vary with their education?

In [ ]:
sns.histplot(data=df,x='Mortgage');
plt.show()

sns.boxplot(data=df,x='Mortgage');
plt.show()

In [ ]:
sns.histplot(data=df,x='CCAvg');
plt.show()

sns.boxplot(data=df,x='CCAvg');
plt.show()

# **Observations:**

The plot is right skewed.
Majority of the customers have a lower average monthly spending on their credit cards.

**Observation:**
The 2 plots above show that the number of people who have not taken a Mortgage are very high.
The plots are heavily right skewed.
Out of 5000 bank customers, 3500 do not have any existing mortgage loan.

In [ ]:
sns.histplot(data=df,x='Income');
plt.show()

sns.boxplot(data=df,x='Income');
plt.show()

# **Observations:**

Income of customers is also right skewed.

In [ ]:
sns.histplot(data=df,x='CreditCard');
plt.show()

**Observation:**
1500 customers have a credit card from the bank.

In [ ]:
sns.histplot(data=df,x='Online');
plt.show()

# **Observation**:

3000/5000 customers utilize Internet Banking Services - which is 60%

In [ ]:
sns.histplot(data=df,x='Personal_Loan');
plt.show()

# **Observations:**

Only 500/5000 customers have a Personal Loan.
The dataset is unabalanced with respect to Personal Loan.


In [ ]:
var = ['Age', 'Experience', 'Income', 'County','Family','CCAvg', 'Education', 'Mortgage', 'Personal_Loan','Securities_Account','CD_Account','Online','CreditCard']
corr = df[var].corr()
plt.figure(figsize=(12,12))
sns.heatmap(corr, annot=True, cmap='Spectral')
plt.show()

**Observation:**
In the given dataset:  

*    Age and Experience have the highest correlation.
*   Correlation is also high between Average spending on credit cards(CCAvg) and Income - which directly implies that spending on credit cards is directly proportional to the income of the customer.

*   **Correlation is also high between Income and Personal Loan.**
*   **Correlation is moderately high between CCAvg and Personal Loan.**

*   Correlation is also moderately high between Personal_Loan/Securities_Account and CD_Account

*   Correlation is moderately high between Credit card and CD_Account - meaning customers who took credit cards from other banks are opening a CD Account with AllLife bank.


**The attributes that have a strong correlation to Personal Loan are:**

*   **Income**
*   **Average spending on Credit cards (CCAvg)**
*   **CD_Account**






**Creating a Boxplot and Histogram combined**

In [ ]:
   #data - DataFrame
   #feature - Column name
   #kde - Kernel Density Estimation - Draws the KDE line if set to True

def histogram_boxplot(data, feature, binwid=None, figsize=(12, 7), kde=True,):

   #Creating a figure with 2 rows for the boxplot and histogram.
    f2, (ax_box2, ax_hist2) = plt.subplots(nrows=2, sharex=True, figsize=figsize)

    # Creating boxplot with a mark to show the mean value.
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="orange"
    )

    #Plotting the histogram
    sns.histplot(
        data=data, x=feature, binwidth=binwid, kde=kde, ax=ax_hist2
    )

    #Adding mean line to histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )

    # Adding median line to histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )

**Plotting non-zero values of Mortgage**

In [ ]:
df_mortgage_nonzero = df[df["Mortgage"] != 0].copy()
histogram_boxplot(df_mortgage_nonzero, "Mortgage", 10)

# **Observation:**
Even when non-zero values of Mortgage are plotted, the graph is right skewed.
Existing customers have mainly taken mortgages between $80,000-$200,000

**Mean** Mortgage amount = $185,000

**Median** is approx $155,000

In [ ]:
histogram_boxplot(df, "Age")

# **Observation:**
Age of customers is evenly distributed.
The closeness of Mean and Median lines also indicates the same - even distribution of age.

In [ ]:
histogram_boxplot(df, "Experience")

# **Observation:**
Years of Experience of customers is alsoevenly distributed.
Mean and median lines are coinciding.

In [ ]:
histogram_boxplot(df, "Income")

# **Observation:**
Income of customers is also right skewed.

Mean income being $75,000

In [ ]:
histogram_boxplot(df, "CCAvg")

# **Observation:**

Average credit card spending per month is right skewed.

Mean being close to $2000. Lot of outliers.

There are 500/5000 customers whose credit card spending per month is 0 - which is 10% of the customer base of AllLife bank.

**Function to create a BarPlot which displays exact Count of each category on the top**

In [ ]:
#Data - DataFrame
#Feature - Name of the column for which the BarPlot is being plotted
#Perc - Set to True if you want to display Percentage on top of each bar instead of the total count.
#n - Specifies the number of bars to display - Top category levels are displayed first.
#If n is NOT passed, all category types in the data are displayed as bars as n is set to default value None in the function definition

def labeled_barplot(data, feature, hue=None, perc=False, n=None):

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
       palette="Paired",
        hue = hue,
        order=data[feature].value_counts().index[:n].sort_values()
        #identifies the n most frequent unique values in a specified column and then sorts those n values in ascending order.
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
labeled_barplot(df, "County", "Personal_Loan", perc=True, n=15)

# **Observation:**

In the top 15 zipcodes where AllLife bank customers are there, there are MORE number of customers who did NOT take a Personal Loan than those whi have taken.

ZipCode 94720 has the highest number of AllLife bank customers.

Followed by zipcodes 94305 and 95616

Rest of the zipcodes have an almost equal number of AllLife bank customers - which is interesting to note.

In [ ]:
labeled_barplot(df, "County", n=20)

# **Observation:**

After plotting the graph before, plotted this graph to be able to observe the number of customers in each county.

In [ ]:
labeled_barplot(df, "Family", "Online",perc=True)

# **Observation:**

For each family size, number of customers who utilize online banking facilites is MORE than those who do not.

Family size of 1 using internet banking facilities the most when compared to other family sizes.

In [ ]:
labeled_barplot(df, "Personal_Loan", "Online")

# **Observation:**

Plotted this graph to check if utilizing internet banking facilities has any correlation to taking a Personal Loan.

Looks like that is NOT the case.

Majority of the customers who have internet banking facility do not have a Personal Loan.
At the same time, only a small amount of customers have a Personal Loan at the bank. Internet banking or in-person banking does not seem to have an impact on taking a Personal Loan.

In the customers who took a Personal Loan, significant amount of them have Internet Banking facility

In [ ]:
labeled_barplot(df, "Personal_Loan", "CD_Account")

# **Observations:**

Having a CD account has little to no impact on the customer taking a Personal Loan.

Majority of customers who took a Personal Loan do not have a CD account

In [ ]:
labeled_barplot(df, "Personal_Loan", "Securities_Account")

# **Observations:**

Having a Securities account has no impact on the Personal Loan.

Majority of customers who took a Personal Loan do not have a CD account.
Very few number of customers who took a Personal Loan have a Securities account

In [ ]:
labeled_barplot(df, "Personal_Loan", "HasMortgage")

# **Observations:**

Having a Mortgage has very little impact on whether the customer takes a  Personal Loan or NOT.

Majority of customers who took a Personal Loan do not have Mortgage.

In [ ]:
sns.boxplot(x='Personal_Loan', y='CCAvg', data=df)
plt.show()

# **Observations:**

The above box plot clearly indicates that customers who have an Average Credit card spending per month greater than 2.5k take a Personal Loan

In [ ]:
labeled_barplot(df, "Personal_Loan", "CreditCard")

# **Observations:**


Having a Credit card from another bank has very little impact on whether the customer takes a  Personal Loan or NOT.

Majority of customers who took a Personal Loan do not have a Credit card from another bank.

In [ ]:
sns.boxplot(x='Personal_Loan', y='HasAllLifeCreditCard', data=df)
plt.show()

labeled_barplot(df, "Personal_Loan", "HasAllLifeCreditCard")

# **Observations:**

Having an All Life Bank Credit Card seems to have very little impact on whether the customer has a personal loan or NOT.

However, among the customers that do have a Personal Loan, approximately half of them also own an All Life Bank credit card

In [ ]:
sns.boxplot(x='Personal_Loan', y='Age', data=df)
plt.show()


# **Observations:**

From the above box plot, Age of the customer does not have a clear impact on whether the customer takes a Personal Loan or not.

In [ ]:
sns.boxplot(x='Personal_Loan', y='Experience', data=df)
plt.show()

# **Observations:**

From the above box plot, Experience of the customer does not have a clear impact on whether the customer takes a Personal Loan or not.

In [ ]:
sns.boxplot(x='Personal_Loan', y='Income', data=df)
plt.show()

# **Observations:**

From the box plot, customers who have an Income >100k took a Personal Loan.
However, there are also some outliers in the no Personal Loan for customers who have an Income above 150k.

It could be said that custoemrs with Income between 100-150k, has a greater chance of taking a Personal Loan.

In [ ]:
sns.boxplot(x='Personal_Loan', y='Family', data=df)
plt.show()

labeled_barplot(df, "Personal_Loan", "Family")

# **Observations:**

Family greater than or equal to 3 has a higher chance of taking a Personal Loan.

In [ ]:
sns.boxplot(x='Personal_Loan', y='County', data=df)
plt.show()

# **Observations:**

County of the customer does NOT seem to have much impact on whether he takes a Personal Loan or NOT.
However, majority of the customers seem to residing in the counties between 920 and 945.

In [ ]:
sns.scatterplot(x='CCAvg', y='Income', hue='Personal_Loan', data=df, s=100)
plt.title('Scatter Plot of CCAvg vs Income with Category as Personal Loan')
plt.xlabel('CCAvg')
plt.ylabel('Income')
plt.legend(title='Personal_Loan')
plt.grid(True)
plt.show()

# **Observations:**

Clearly income above 100k customers have a greater chance of taking a Personal Loan.

Clean split can be drawn at Income = 100k
However, CCAvg data is spread out across the horizontal axis.

A split can also be drawn at CCAvg=4 to divide the customrs who have and do NOT have a Personal loan.

In [ ]:
sns.scatterplot(x='CCAvg', y='Family', hue='Personal_Loan', data=df, s=100)
plt.title('Scatter Plot of CCAvg vs Family with Category as Personal Loan')
plt.xlabel('CCAvg')
plt.ylabel('Family')
plt.legend(title='Personal_Loan')
plt.grid(True)
plt.show()

# **Observations:**

For Average monthly credit card spending less than 3K, with any number of members in the family, customers have NOT taken a Personal Loan.

For average monthly credit card spending >3k, families of 3 and 4 have Personal Loans.

In [ ]:
sns.scatterplot(x='Income', y='Family', hue='Personal_Loan', data=df, s=100)
plt.title('Scatter Plot of Income vs Family with Category as Personal Loan')
plt.xlabel('Income')
plt.ylabel('Family')
plt.legend(title='Personal_Loan')
plt.grid(True)
plt.show()

# **Observations:**

For Income >100k, with any number of members in the family, customers have NOT taken a Personal Loan.

For Income >100k, families of 3 and 4 have Personal Loans.

In [ ]:
labeled_barplot(df, "Securities_Account", "Online")

# **Observation:**

Checked the same for Securities Account.
Utilizing online banking facilities has Nothing much to do with whether the customer will open a Securities Account.

Internet banking or in-person banking does not seem to have an impact on opening a Securities Account.

In [ ]:
labeled_barplot(df, "CD_Account", "Online")

# **Observation:**

Internet banking or in-person banking does not seem to have much of an impact on opening a CD account.

However, it looks like among the CD account holders, a majority of them use internet banking.

In [ ]:
labeled_barplot(df, "HasMortgage", "Online")

# **Observation:**

Majority of the customers do NOT have Mortgage.

Among the customers who took Mortgage from AllLife Bank, people who use internet Banking seem to have a slightly higher tendency to take Mortgage or vice versa - people who took Mortgage have a tendency to start using Internet Banking Services.

In [ ]:
labeled_barplot(df, "HasMortgage", "Securities_Account")

# **Observation:**

Customers who have Mortgage tend to NOT have a Securities Account with AllLife Bank.

In [ ]:
labeled_barplot(df, "HasMortgage", "Family")

# **Observation:**

Customers who are family of 1,2 have a greater tendency to take a Mortgage than customers who are a family of 3 or 4.

In [ ]:
labeled_barplot(df, "Education", "HasMortgage")

# **Observation: **

Education is NOT playing a role on whether a customer decides to take a Mortgage or NOT.

Percentage of undergraduates who took Mortgage, percentage of graduates who took Mortgage and percentage of Professionals who took Mortgage is almost similar (approx. 45%)


In [ ]:
labeled_barplot(df, "Education", "Online")

# **Observation:**

More number of customers who are undergraguates have a higher tendency to use Internet Banking services than graduates and professionals.

In [ ]:
labeled_barplot(df, "Family", perc=True)

# **Observation:**

AllLife Bank has more customers who are a family of 1, followed by customers who are a family of 2 and 3.

In [ ]:
labeled_barplot(df, "Education","Securities_Account")

# **Observation:**

Even though the number of undergraduates are more, the percentage of undergraduates who hold a Securities account is almost same as the percentage of Graduates who hold a Securities account and the percentage of professionals who hold a Securities account - approx 11% each

In [ ]:
labeled_barplot(df, "Education")

# **Observation:**

Customers of AllLife bank are mostly undergraduates, followed by Professionals and then graduates coming a close third.

In [ ]:
labeled_barplot(df, "Education","CD_Account")

# **Observation:**

Even though the number of undergraduate customers are more, the percentage of undergraduates who hold a CD account are less when compared to the percentage of graduates and professionals that hold a CD account.

In [ ]:
labeled_barplot(df, "Education","Personal_Loan")

# **Observation: **

The bank has more customers who are undergraduates than graduates and professionals.

Percentage of undergraduates who took a Personal loan are less(approx 4.5%) when compared to the percentage of Graduates and Professionals who took a personal loan (approx 16% each)

Based on the above plot, customers who have a higher eductaion are more inclined to purchase a loan.

The bank can look at ways to tap into potential customers who are graduates and professionals.

In [ ]:
labeled_barplot(df, "Securities_Account", "Personal_Loan")

# **Observation:**

Customers who did NOT have a Securities account also did NOT take a Personal Loan.

Customers who have a Securities account usually did NOT take a Personal Loan.

In [ ]:
labeled_barplot(df, "Securities_Account", "CD_Account")

# **Observation:**

Mostly, Customers who did NOT have a Securities account also did NOT have a CD account.

When compared between CD Account and Personal Loan, customers who have a Securities Account are more likely to also have a CD Account than take a Personal loan.

In [ ]:
labeled_barplot(df, "Age")

# **Observation:**
The age range of customers between 28-63 is almost equally distributed.

In [ ]:
labeled_barplot(df, "Age", "Personal_Loan")

# **Observation:**

The number of customers who took a personal loan in each age is almost equal.
Age is Not playing a role in whether a customer is deciding to take a Personal Loan or not.


In [ ]:
labeled_barplot(df, "CD_Account",perc=True)

# **Observation:**

Number of customers who own CDs in the bank are very very less.

In [ ]:
labeled_barplot(df, "Online")

# **Observation:**

More than half of the customers are preferring to use Internet Banking services.

In [ ]:
labeled_barplot(df, "CreditCard", "Online", perc=True)

# **Observation: **

Only 29% of AllLife bank customers own Credit cards.

This can be improved.

In [ ]:
labeled_barplot(df, "Family","CreditCard")



Families of 2 have the highest percentage of customers who have credit cards, follwed by familes of 4,3 and 1.

In [ ]:
labeled_barplot(df, "CCAvg", "HasAllLifeCreditCard", n=25)

# **Observation:**

**CCAvg:** Average spending on credit cards per month (in thousand dollars)

**CreditCard:** Does the customer use a credit card issued by any other Bank (excluding All life Bank)? (0: No, 1: Yes)

Customers of AllLife Bank who have an average Credit spending per month, majorly are using All Life Bank credit cards alone when compared with their spending on other bank Credit cards.

Inspite of the above there is a possibility of diverting the usage of credit cards completely to All Life Bank so that their average spending per month compltely comes on All Life credit card.

another observation is that majority of the customers are in a lower average spending per month on credit cards. Providing incentives when a certain average spending is crossed will also encourage customers to use their credit cards more.

In [ ]:
# Function to plot distributions for each value of target - Personal Loan, CD, Has Mortgage, HasAllLifeCreditCard etc.

def distribution_plot_wrt_target(data, predictor, target):

  #Creating room for 4 sub-plots - 1 row and 2 columns
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

#Gets all the unique values that the target variable can hold
#In this function, targetting ONLY target variables which hold 2 unique values - 0 and 1
    target_uniq = data[target].unique()


    axs[0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0],
        color="teal",
        stat="density",
    )

    axs[1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[1],
        color="orange",
        stat="density",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
distribution_plot_wrt_target(df, "Age", "Personal_Loan")

# **Observations:**

Peak age of customers who took a personal loan is at 35 years.
Peak age of customers who DO NOT have a personal loan is 58 years.

Density of customers who have a personal loan is almost equally distributed across age groups.

In [ ]:
distribution_plot_wrt_target(df, "Experience", "Personal_Loan")

# **Observations:**

Peak experience of customers who took a personal loan is between 5-12years.

Density of customers who have a personal loan is almost equally distributed across different years of experience.

In [ ]:
distribution_plot_wrt_target(df, "Income", "Personal_Loan")

# **Observations:**

Income peak of customers who took personal loan is at 130k, followed by 170-190k
Income peak of customers who did not take personal loan is at 42-46k.

For customers who took personal loan, data is left skewed.
For customers who did not take personal loan, data is right skewed.

In [ ]:
distribution_plot_wrt_target(df, "CCAvg", "Personal_Loan")

# **Observations:**

Peak credit card spending per month for customers who took Personal loan is at 2700$-3400$

Peak credit card spending per month for customers who did not take Personal loan is $0

In [ ]:
distribution_plot_wrt_target(df_mortgage_nonzero, "Mortgage", "Personal_Loan") # df_mortgage_nonzero is the data set which contains all non-zero mortgages.

# **Observations:**

Peak mortgage value taken by customers who have a Personal loan is between 200-300k.

Peak mortgage value taken by customers who did NOT take Personal loan is at 100k.

Data is skewed to the right for customers who did not take Personal Loan.

In [ ]:
distribution_plot_wrt_target(df_mortgage_nonzero, "County", "Personal_Loan")

# **Observations:**

920, 940, 950 and 900 counties have more customers who have a Personal Loan.
Number of customers who do not have a Personal Loan are also more in these counties.

## Data Preprocessing

* Missing value treatment
* Feature engineering (if needed)
* Outlier detection and treatment (if needed)
* Preparing data for modeling
* Any other preprocessing steps (if needed)

In [ ]:
#Checking for missing values in dataset
data.isnull().sum()

# **Observations:**

**There are NO missing values in this dataset.**


In [ ]:
#Check for duplicate values
df.duplicated().sum()

# **Observations:**

**There are NO duplicate values in the data provided.**

In [ ]:
# Identifying if there are any negative values in the columns that should not hold negative values

negative_counts = (df < 0).sum()
print("Count of negative values per column:\n", negative_counts)

In [ ]:
#Converting all negative values in Experience column to absolute values
df['Experience'] = df['Experience'].abs()

In [ ]:
negative_counts = (df < 0).sum()
print("Count of negative values per column:\n", negative_counts)

In [ ]:
# Outlier detection using boxplot
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(15, 12))

for i, variable in enumerate(numeric_columns):
    plt.subplot(4, 4, i + 1)
    plt.boxplot(df[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

# **Observations:**

There are outliers in Income, CCAvg and Mortgage columns.

The outliers in Mortgage column is due to more number of customers not having mortgage which is pushing the customers with Mortgage values towards outlier section. This is accurate data and need not be corrected.

Same is the case with CCAvg. There are more customers whose monthly average credit card spending is either 0 or low. The plot is right skewed. Hence, the data is accurate and will help in arriving at conclusions for the bank. Outliers need not be removed.

For income, there are more customers in the 25-100k income range. There are considerable amount of customers with income >100k and they also have valuable information associated with them which defines decisions for the bank on how to increase the Mortgage, Loan, CD accounts. Hence, the outliers should not be removed.

## Model Building and Model Performance Improvement

In [ ]:
# Drop the original Personal_Loan column to create the feature set (X)
x = df.drop(["Personal_Loan"], axis=1)
y = df["Personal_Loan"]

# Create dummy variables for categorical columns to convert them to numerical data.
# Drop first = TRUE to drop the original column after adding the dummy columns.
X = pd.get_dummies(x, columns=["County"], drop_first=True)

# Splitting data 70:30 to create train and test sets respectively. Stratify =y is ensuring balanced distribution of variables between train and test sets
#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, test_size=0.30, stratify = y, random_state=1
#)

#For this specific imbalanced dataset, removing stratify=y is giving better values for Recall, Precision and F1 score.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1
)

print("Number of rows in Training set : ", X_train.shape[0])
print("Number of rows in Testing set : ", X_test.shape[0],'\n')
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True),'\n')
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

#X_train.head()
#print(np.isnan(X_train).any())

In [ ]:
X_train.head()

# **Observations:**

In train set, percentage of customers without Personal loan is 90% and percentage of customers with Personal Loan is 9.5%
Approximately, Similar stats are maintained in the test set too.

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

In [ ]:
#Model 0 - No pruning

model0 = DecisionTreeClassifier(random_state=1)
model0.fit(X_train, y_train)

In [ ]:
confusion_matrix_sklearn(model0, X_train, y_train)

In [ ]:
decision_tree_default_perf_train = model_performance_classification_sklearn(
    model0, X_train, y_train
)
decision_tree_default_perf_train

In [ ]:
confusion_matrix_sklearn(model0, X_test, y_test)

**For the test set of Model 0:**

Recall:
17/149 times the model made a mistake in predicting.

Precision:
19/151 times the model made a mistake.

In [ ]:
decision_tree_default_perf_test = model_performance_classification_sklearn(
    model0, X_test, y_test
)
decision_tree_default_perf_test

# **Observations:**

There is a difference between test and training scores for Model 0.
The noise in the training set is NOT suiting the noise in the test set maybe.

This model 0 is overfitting.

In [ ]:
# Plotting the decision tree for Model 0
feature_names = list(X_train.columns)

plt.figure(figsize=(20, 10))
out = tree.plot_tree(
    model0,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
# Text report showing the rules of a decision tree -
print(tree.export_text(model0, feature_names=feature_names, show_weights=True))

# **Observations:**

First split of the Model 0 tree is on:
**Income <= 116.50**

In [ ]:
#Model 1 - Balanced decision tree. No pruning

model1 = DecisionTreeClassifier(random_state=1, class_weight="balanced")
model1.fit(X_train, y_train)

In [ ]:
confusion_matrix_sklearn(model1, X_train, y_train)

In [ ]:
decision_tree_perf_train = model_performance_classification_sklearn(
    model1, X_train, y_train
)
decision_tree_perf_train

In [ ]:
confusion_matrix_sklearn(model1, X_test, y_test)

In [ ]:
decision_tree_perf_test = model_performance_classification_sklearn(
    model1, X_test, y_test
)
decision_tree_perf_test

In [ ]:
decision_tree_perf_test = model_performance_classification_sklearn(
    model0, X_test, y_test
)
decision_tree_perf_test

# **Observations:**

Recall, precision and F1 score came down in Model 1 when compared to Model 0

In [ ]:
#Plotting the decision tree for Balanced approach - Model 1

plt.figure(figsize=(20, 10))
out = tree.plot_tree(
    model0,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
#Decision Tree - Pre-pruning

# Define the parameters of the tree to iterate over
max_depth_values = np.arange(2, 7, 2) # Max depth from 2-11 with a step size of 2.
max_leaf_nodes_values = np.arange(10, 51, 20)
min_samples_split_values = np.arange(10, 51, 20)

#max_leaf_nodes_values = [50, 75, 150, 250]
#min_samples_split_values = [10, 30, 50, 70]

# Initialize variables to store the best model and its performance
best_estimator = None
best_score_diff = float('inf')
best_test_score = 0.0

# Iterate over all combinations of the specified parameter values
for max_depth in max_depth_values:
    for max_leaf_nodes in max_leaf_nodes_values:
        for min_samples_split in min_samples_split_values:

            # Initialize the tree with the current set of parameters
            estimator = DecisionTreeClassifier(
                max_depth=max_depth,
                max_leaf_nodes=max_leaf_nodes,
                min_samples_split=min_samples_split,
                #class_weight='balanced',
                random_state=1
            )

            # Fit the model to the training data
            estimator.fit(X_train, y_train)

             # Make predictions on the training and test sets
            y_train_pred = estimator.predict(X_train)
            y_test_pred = estimator.predict(X_test)

            # Calculate recall scores for training and test sets
            train_recall_score = recall_score(y_train, y_train_pred)
            test_recall_score = recall_score(y_test, y_test_pred)

            # Calculate F1 scores for training and test sets
            train_f1_score = f1_score(y_train, y_train_pred)
            test_f1_score = f1_score(y_test, y_test_pred)

            # Calculate the absolute difference between training and test F1 scores
            score_diff = abs(train_f1_score - test_f1_score)

            # Update the best estimator and best score if the current one has a smaller score difference
            #if (score_diff < best_score_diff) & (test_f1_score > best_test_score):
            if (score_diff < best_score_diff):
                best_score_diff = score_diff
                best_test_score = test_f1_score
                best_estimator = estimator

# Print the best parameters
print("Best parameters found:")
print(f"Max depth: {best_estimator.max_depth}")
print(f"Max leaf nodes: {best_estimator.max_leaf_nodes}")
print(f"Min samples split: {best_estimator.min_samples_split}")
print(f"Best test F1 score: {best_test_score}")

In [ ]:
# creating an instance of the best model identified from above
model2 = best_estimator

# fitting the best model to the training data
model2.fit(X_train, y_train)

In [ ]:
#Confusion matrix of training data

confusion_matrix_sklearn(model2, X_train, y_train)

In [ ]:
decision_tree_tune_perf_train = model_performance_classification_sklearn(
    model2, X_train, y_train
)
decision_tree_tune_perf_train

In [ ]:
confusion_matrix_sklearn(model2, X_test, y_test)

In [ ]:
decision_tree_tune_perf_test = model_performance_classification_sklearn(
    model2, X_test, y_test
)
decision_tree_tune_perf_test

# **Observations:**

**For the pre-pruned model:**

F1 and Recall scores have reduced for the test data when compared with the train data

In [ ]:
feature_names = list(X_train.columns)
#importances = model2.feature_importances_
#indices = np.argsort(importances)

In [ ]:
plt.figure(figsize=(20, 10))
out = tree.plot_tree(
    model2,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
# below code will add arrows to the decision tree split if they are missing
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
# Text report showing the rules of a decision tree -
print(tree.export_text(model2, feature_names=feature_names, show_weights=True))

# **Observations:**

The above pre-pruned tree gives us the following information:

*   If CCAvg > 2.95 and CD_Account is there, customer will take a Personal Loan - Class prediction is 1
*   If Income > 92.5k and Education is Graduate or Professional, customer will take a Personal Loan

*   If Income > 116.5 and Education is undergraduate and Family size > 3, customer will take a Personal Loan
*   If Income > 116.5 and Education is graduate or Professional, customer will take a Personal Loan





In [ ]:
# importance of features in the tree building

importances = model2.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(40, 8))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

# **Observation:**

Education, Income, Family, CCAvg are the important features that define better prediction.

In [ ]:
#Model 4 - Decision tree - Post-pruning

clf = DecisionTreeClassifier(random_state=1, class_weight="balanced") #class_weight allows to balance the current imbalanced dataset

#cost complexity
path = clf.cost_complexity_pruning_path(X_train, y_train)

#Extracting array of effective alphas from the pruning path
#Extract array of total impurities for each alpha along the pruning path
ccp_alphas, impurities = abs(path.ccp_alphas), path.impurities

In [ ]:
pd.DataFrame(path)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")
plt.show()

In [ ]:
#List for storing the classifiers
clfs = []

for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(
        random_state=1, ccp_alpha=ccp_alpha, class_weight="balanced"
    )
    clf.fit(X_train, y_train)
    clfs.append(clf)

print(
    "Number of nodes in the last tree is: {} with ccp_alpha: {}".format(
        clfs[-1].tree_.node_count, ccp_alphas[-1]
    )
)

In [ ]:
# Removing the last classifier and corresponding ccp_alpha value from the list
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

#Extract the number of nodes
node_counts = [clf.tree_.node_count for clf in clfs]

#Extract maximum depth of each classifier
depth = [clf.tree_.max_depth for clf in clfs]

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(ccp_alphas, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_xlabel("alpha")
ax[0].set_ylabel("number of nodes")
ax[0].set_title("Number of nodes vs alpha")
ax[1].plot(ccp_alphas, depth, marker="o", drawstyle="steps-post")
ax[1].set_xlabel("alpha")
ax[1].set_ylabel("depth of tree")
ax[1].set_title("Depth vs alpha")
fig.tight_layout()

In [ ]:
recall_train = []

#Iterate through each decision tree classifier in clfs
for clf in clfs:
    pred_train = clf.predict(X_train)
    values_train = recall_score(y_train, pred_train)
    recall_train.append(values_train)

In [ ]:
recall_test = []
for clf in clfs:
    pred_test = clf.predict(X_test)
    values_test = recall_score(y_test, pred_test)
    recall_test.append(values_test)

In [ ]:
train_scores = [clf.score(X_train, y_train) for clf in clfs]
test_scores = [clf.score(X_test, y_test) for clf in clfs]

In [ ]:
#Using highest Recall score as the defining parameter for tree pruning since for this dataset to be able to predict closely
#if the customer will take a Personal Loan or not, we want to ensure that the number of false negatives are minimised - meaning we DO NOT want to predict negative
#when the customer will in reality take a Personal loan. We want to minimise missing customers due to a low Recall.


fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlabel("alpha")
ax.set_ylabel("Recall")
ax.set_title("Recall vs alpha for training and testing sets")
ax.plot(
    ccp_alphas, recall_train, marker="o", label="train", drawstyle="steps-post",
)
ax.plot(ccp_alphas, recall_test, marker="o", label="test", drawstyle="steps-post")
ax.legend()
plt.show()

In [ ]:
# creating the model where we get highest train and test recall
index_best_model = np.argmax(recall_test)
best_model = clfs[index_best_model]
print(best_model)

In [ ]:
model4 = best_model
confusion_matrix_sklearn(model4, X_train, y_train)

In [ ]:
decision_tree_post_perf_train = model_performance_classification_sklearn(
    model4, X_train, y_train
)
decision_tree_post_perf_train

In [ ]:
confusion_matrix_sklearn(model4, X_test, y_test)

In [ ]:
decision_tree_post_test = model_performance_classification_sklearn(
    model4, X_test, y_test
)
decision_tree_post_test

# **Observations:**

**For the post-pruned model:**

Difference in performance between train and test is little for all values - Accuracy, Recall, Precision and F1

Test and training scores are the closest - which implies that overfitting has been controlled in Post-pruned decision tree.

In [ ]:
plt.figure(figsize=(20, 10))

out = tree.plot_tree(
    model4,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=False,
    class_names=None,
)
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
# Text report showing the rules of a decision tree -

print(tree.export_text(model4, feature_names=feature_names, show_weights=True))

## Model Performance Comparison and Final Model Selection

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [
        decision_tree_default_perf_train.T,
        decision_tree_perf_train.T,
        decision_tree_tune_perf_train.T,
        decision_tree_post_perf_train.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Decision Tree (sklearn default)",
    "Decision Tree with class_weight",
    "Decision Tree (Pre-Pruning)",
    "Decision Tree (Post-Pruning)",
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
# testing performance comparison

models_test_comp_df = pd.concat(
    [
        decision_tree_default_perf_test.T,
        decision_tree_perf_test.T,
        decision_tree_tune_perf_test.T,
        decision_tree_post_test.T,
    ],
    axis=1,
)
models_test_comp_df.columns = [
    "Decision Tree (sklearn default)",
    "Decision Tree with class_weight",
    "Decision Tree (Pre-Pruning)",
    "Decision Tree (Post-Pruning)",
]
print("Test set performance comparison:")
models_test_comp_df

# **Observations:**

Training performance and test performance is on similar lines for all factors for pre-pruned and post-pruned decision trees.
So, the 2 models are suitable to consider for evaluation.


Considering that we want the Recall to be high for identifying customers that take a Personal Loan and reduce the number of false negatives, developed the post-pruned model based on high Recall.

However, the precision and F1 score have dropped considerably.

On the other hand for the pre-pruned tree, the Precision and F1 score are very high and the Recall score is also NOT bad at 86.5%

**So, we should move ahead with the pre-pruned decision tree as the model**


## Actionable Insights and Business Recommendations


* What recommedations would you suggest to the bank?

*   The bank can deploy the pre-pruned model to predict whether an existing customer will take a Personal Loan or not.
*   The bank should alsp advertise more among existing as well as probable customers who are considered suitable by the pre-pruned Decision tree.


*   Customers do not seem to be using Internet Banking Services to open Personal Loan, CD or open a Securities Account.
The process of opening Securities Account, Personal Loan or CD should be made easier and the process must be advertised via phone, media etc. so that more customers are able to open and use these accounts using Internet banking services.

*   Customers who have a higher eductaion are more inclined to take a Personal loan, open a CD account or a Securities account.
The bank can look at ways to tap into more potential customers who are graduates and professionals to open one of these accounts.

*   For Average monthly credit card spending less than 3K, with any number of members in the family, customers have NOT taken a Personal Loan.

*   For average monthly credit card spending >3k, families of 3 and 4 have Personal Loans. Bank can advertise more and communicate more to such customers regarding Personal Loan facilities.

*   The pre-pruned tree gives us the following information which the bank can use when communicating to existing as well as new customers:

If CCAvg > 2.95 and CD_Account is there, customer will take a Personal Loan.

If Income > 92.5k and Education is Graduate or Professional, customer will take a Personal Loan

If Income > 116.5 and Education is undergraduate and Family size > 3, customer will take a Personal Loan

If Income > 116.5 and Education is graduate or Professional, customer will take a Personal Loan


*   Income above 100k customers have a greater chance of taking a Personal Loan.









___